# Run the Data Wrangler `taxi.flow` headlessly (no UI import)

Executes a Data Wrangler flow as a **SageMaker Processing job** on the managed
Data Wrangler container — the same thing the DW UI's *Export → Processing job*
produces. Run this from a Studio/Unified Studio notebook or anywhere with the SDK.

_Caveat: `taxi.flow` here is a hand-authored scaffold. The mechanism below is the
standard one; for a guaranteed run, use a real DW-exported flow. The identical
transforms also run for free via `shared/taxi_transforms.py` (Glue/EMR/notebook)._

In [ ]:
import json, sagemaker
from sagemaker import image_uris
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()   # in a project notebook this is automatic
bucket = "roi-smdemo-029331796573-us-east-2"

# Upload the flow, then find its final output node ('<node_id>.default').
flow_key = "dw/taxi.flow"
sess.boto_session.client("s3").upload_file("taxi.flow", bucket, flow_key)
flow_s3 = f"s3://{bucket}/{flow_key}"
with open("taxi.flow") as f:
    output_name = json.load(f)["nodes"][-1]["node_id"] + ".default"
print(flow_s3, output_name)

In [ ]:
# Managed Data Wrangler container image for this region (SDK resolves the URI).
image = image_uris.retrieve(framework="data-wrangler", region=region)

processor = Processor(
    role=role, image_uri=image, instance_count=1,
    instance_type="ml.m5.4xlarge", volume_size_in_gb=30,
    sagemaker_session=sess, base_job_name="roi-smdemo-dw-flow",
)

flow_input = ProcessingInput(
    source=flow_s3, destination="/opt/ml/processing/flow", input_name="flow",
    s3_data_type="S3Prefix", s3_input_mode="File", s3_data_distribution_type="FullyReplicated",
)
output = ProcessingOutput(
    output_name=output_name, source="/opt/ml/processing/output",
    destination=f"s3://{bucket}/processed/data-wrangler/", s3_upload_mode="EndOfJob",
)

processor.run(
    inputs=[flow_input], outputs=[output],
    arguments=["--output-config", json.dumps({output_name: {"content_type": "CSV"}})],
    wait=True, logs=True,
)
print("output ->", f"s3://{bucket}/processed/data-wrangler/")